In [ ]:
# ftcircuitbench/pbc_converter/pbc_circuit_reader.py
"""
Functions to read and parse combined PBC files generated by combine_pbc_files.
"""

In [ ]:
import os
import re
from typing import Dict, List, Tuple

In [ ]:
def read_combined_pbc_file(filepath: str) -> Dict[str, any]:
    """
    Reads a combined PBC file and parses both T-layers and measurement basis sections.

    Args:
        filepath (str): Path to the combined PBC file (e.g., PBC_pre_opt.txt)

    Returns:
        Dict containing:
            - 't_layers': List of layers, each containing list of Pauli strings
            - 'measurement_basis': List of Pauli strings for measurement
            - 'filepath': Original filepath
            - 'sections_found': List of section names found
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"PBC file not found: {filepath}")

    result = {
        "t_layers": [],
        "measurement_basis": [],
        "filepath": filepath,
        "sections_found": [],
    }

    current_section = None
    current_layer = None

    with open(filepath, "r") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()

            # Skip empty lines
            if not line:
                continue

            # Check for section headers (combined format)
            if line.startswith("--- T-Layers"):
                current_section = "t_layers"
                result["sections_found"].append("t_layers")
                continue
            elif line.startswith("--- Measurement Basis"):
                current_section = "measurement_basis"
                result["sections_found"].append("measurement_basis")
                continue
            elif line.startswith("---"):
                # End of section
                current_section = None
                continue

            # Parse content based on current section
            if current_section == "t_layers":
                if line.startswith("LAYER"):
                    # Start of a new layer
                    if current_layer is not None:
                        result["t_layers"].append(current_layer)
                    current_layer = []
                elif line.startswith("  ") and current_layer is not None:
                    # Pauli string in current layer (remove leading spaces)
                    pauli_str = line.strip()
                    if pauli_str and pauli_str != "(empty layer)":
                        current_layer.append(pauli_str)
                elif line == "(empty layer)" and current_layer is not None:
                    # Empty layer - keep it as empty list
                    pass

            elif current_section == "measurement_basis":
                # Measurement basis lines are direct Pauli strings
                if line and line != "NO MEASUREMENT BASIS":
                    result["measurement_basis"].append(line)

    # Don't forget the last layer
    if current_layer is not None:
        result["t_layers"].append(current_layer)

    # If no sections were found, try to parse as individual file format
    if not result["sections_found"]:
        return read_individual_pbc_file(filepath)

    return result

In [ ]:
def read_individual_pbc_file(filepath: str) -> Dict[str, any]:
    """
    Reads an individual PBC file (either tlayers.txt or measure_basis.txt).

    Args:
        filepath (str): Path to the individual PBC file

    Returns:
        Dict containing parsed data
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"PBC file not found: {filepath}")

    result = {
        "t_layers": [],
        "measurement_basis": [],
        "filepath": filepath,
        "sections_found": [],
    }

    filename = os.path.basename(filepath).lower()

    with open(filepath, "r") as f:
        lines = f.readlines()  # DO NOT STRIP HERE

    # Determine file type based on filename
    if "tlayers" in filename:
        result["sections_found"].append("t_layers")
        current_layer = None

        for line in lines:
            if not line.strip():
                continue

            if line.startswith("LAYER"):
                if current_layer is not None:
                    result["t_layers"].append(current_layer)
                current_layer = []
            elif line.startswith("  ") and current_layer is not None:
                pauli_str = line.strip()
                if pauli_str and pauli_str != "(empty layer)":
                    current_layer.append(pauli_str)
            elif line.strip() == "(empty layer)" and current_layer is not None:
                # Empty layer - keep it as empty list
                pass

        # Don't forget the last layer
        if current_layer is not None:
            result["t_layers"].append(current_layer)

    elif "measure_basis" in filename:
        result["sections_found"].append("measurement_basis")
        for line in lines:
            if line.strip() and line.strip() != "NO MEASUREMENT BASIS":
                result["measurement_basis"].append(line.strip())

    return result

In [ ]:
def parse_pauli_string(pauli_str: str) -> Dict[str, any]:
    """
    Parses a Pauli string to extract information about the operator.

    Args:
        pauli_str (str): Pauli string like "+X0Z1" or "-Y2" or "+IXII"

    Returns:
        Dict containing:
            - 'pauli_type': 'X', 'Y', or 'Z'
            - 'qubit_indices': List of qubit indices
            - 'weight': Number of non-identity Paulis
            - 'raw_string': Original string
            - 'sign': '+' or '-'
    """
    # Handle different formats
    # Format 1: "+X0Z1" (with qubit indices)
    # Format 2: "+IXII" (with I for identity, no indices)

    # Extract sign
    sign = "+"
    if pauli_str.startswith("+") or pauli_str.startswith("-"):
        sign = pauli_str[0]
        pauli_str = pauli_str[1:]

    # Try format 1: with qubit indices
    pattern = r"([XYZ])(\d+)"
    matches = re.findall(pattern, pauli_str)

    if matches:
        # Format 1: "+X0Z1"
        pauli_types = [match[0] for match in matches]
        qubit_indices = [int(match[1]) for match in matches]
        weight = len(pauli_types)
    else:
        # Format 2: "+IXII"
        pauli_types = []
        qubit_indices = []
        for i, char in enumerate(pauli_str):
            if char in ["X", "Y", "Z"]:
                pauli_types.append(char)
                qubit_indices.append(i)
        weight = len(pauli_types)

    return {
        "pauli_type": pauli_types[0] if pauli_types else None,
        "qubit_indices": qubit_indices,
        "weight": weight,
        "raw_string": pauli_str,
        "sign": sign,
    }

In [ ]:
def analyze_pbc_file_content(pbc_data: Dict[str, any]) -> Dict[str, any]:
    """
    Analyzes the content of a parsed PBC file to extract statistics.

    Args:
        pbc_data (Dict): Output from read_combined_pbc_file or read_individual_pbc_file

    Returns:
        Dict containing analysis statistics
    """
    analysis = {
        "num_t_layers": len(pbc_data["t_layers"]),
        "num_measurement_operators": len(pbc_data["measurement_basis"]),
        "t_layer_sizes": [],
        "pauli_weight_distribution": {},
        "qubit_interaction_degree": {},
        "total_pauli_operators": 0,
        "sign_distribution": {"+": 0, "-": 0},
    }

    # Analyze T-layers
    for layer_idx, layer in enumerate(pbc_data["t_layers"]):
        analysis["t_layer_sizes"].append(len(layer))
        analysis["total_pauli_operators"] += len(layer)

        for pauli_str in layer:
            parsed = parse_pauli_string(pauli_str)
            weight = parsed["weight"]
            sign = parsed["sign"]

            analysis["pauli_weight_distribution"][weight] = (
                analysis["pauli_weight_distribution"].get(weight, 0) + 1
            )
            analysis["sign_distribution"][sign] += 1

            # Track qubit interactions
            for qubit_idx in parsed["qubit_indices"]:
                analysis["qubit_interaction_degree"][qubit_idx] = (
                    analysis["qubit_interaction_degree"].get(qubit_idx, 0) + 1
                )

    # Analyze measurement basis
    for pauli_str in pbc_data["measurement_basis"]:
        parsed = parse_pauli_string(pauli_str)
        weight = parsed["weight"]
        sign = parsed["sign"]

        analysis["pauli_weight_distribution"][weight] = (
            analysis["pauli_weight_distribution"].get(weight, 0) + 1
        )
        analysis["sign_distribution"][sign] += 1
        analysis["total_pauli_operators"] += 1

        # Track qubit interactions
        for qubit_idx in parsed["qubit_indices"]:
            analysis["qubit_interaction_degree"][qubit_idx] = (
                analysis["qubit_interaction_degree"].get(qubit_idx, 0) + 1
            )

    return analysis

In [ ]:
def validate_pbc_file(filepath: str) -> Tuple[bool, List[str]]:
    """
    Validates that a PBC file can be read and parsed correctly.

    Args:
        filepath (str): Path to the PBC file to validate

    Returns:
        Tuple of (is_valid, list_of_errors)
    """
    errors = []

    try:
        # Try to read the file
        pbc_data = read_combined_pbc_file(filepath)

        # Check that we found expected sections
        if not pbc_data["sections_found"]:
            errors.append("No sections found in file")

        # Check that at least one section was found
        if not pbc_data["sections_found"]:
            errors.append("No valid sections found in file")

        # Try to analyze the content
        try:
            analyze_pbc_file_content(pbc_data)
        except Exception as e:
            errors.append(f"Error analyzing content: {str(e)}")

        # Check for basic structure
        if pbc_data["t_layers"]:
            for i, layer in enumerate(pbc_data["t_layers"]):
                for j, pauli_str in enumerate(layer):
                    try:
                        parsed = parse_pauli_string(pauli_str)
                        if parsed["weight"] == 0:
                            errors.append(
                                f"Layer {i}, operator {j}: Zero-weight Pauli string: {pauli_str}"
                            )
                    except Exception as e:
                        errors.append(
                            f"Layer {i}, operator {j}: Cannot parse Pauli string '{pauli_str}': {str(e)}"
                        )

        # Check measurement basis
        for i, pauli_str in enumerate(pbc_data["measurement_basis"]):
            try:
                parsed = parse_pauli_string(pauli_str)
                if parsed["weight"] == 0:
                    errors.append(
                        f"Measurement operator {i}: Zero-weight Pauli string: {pauli_str}"
                    )
            except Exception as e:
                errors.append(
                    f"Measurement operator {i}: Cannot parse Pauli string '{pauli_str}': {str(e)}"
                )

    except FileNotFoundError:
        errors.append(f"File not found: {filepath}")
    except Exception as e:
        errors.append(f"Error reading file: {str(e)}")

    return len(errors) == 0, errors

In [ ]:
def print_pbc_file_summary(filepath: str):
    """
    Prints a summary of a PBC file's content.

    Args:
        filepath (str): Path to the PBC file
    """
    try:
        pbc_data = read_combined_pbc_file(filepath)
        analysis = analyze_pbc_file_content(pbc_data)

        print(f"=== PBC File Summary: {os.path.basename(filepath)} ===")
        print(f"File path: {filepath}")
        print(f"Sections found: {', '.join(pbc_data['sections_found'])}")
        print()

        print("T-Layers:")
        print(f"  Number of layers: {analysis['num_t_layers']}")
        print(f"  Total T-operators: {sum(analysis['t_layer_sizes'])}")
        if analysis["t_layer_sizes"]:
            print(f"  Layer sizes: {analysis['t_layer_sizes']}")
            print(
                f"  Average layer size: {sum(analysis['t_layer_sizes']) / len(analysis['t_layer_sizes']):.2f}"
            )
        print()

        print("Measurement Basis:")
        print(f"  Number of operators: {analysis['num_measurement_operators']}")
        print()

        print("Pauli Weight Distribution:")
        for weight, count in sorted(analysis["pauli_weight_distribution"].items()):
            print(f"  Weight {weight}: {count} operators")
        print()

        print("Sign Distribution:")
        for sign, count in analysis["sign_distribution"].items():
            print(f"  {sign}: {count} operators")
        print()

        print("Qubit Interaction Degree:")
        for qubit_idx, count in sorted(analysis["qubit_interaction_degree"].items()):
            print(f"  Qubit {qubit_idx}: {count} interactions")
        print()

        print(f"Total Pauli operators: {analysis['total_pauli_operators']}")

    except Exception as e:
        print(f"Error reading PBC file: {e}")

In [ ]:
if __name__ == "__main__":
    import sys

    if len(sys.argv) < 2:
        print("Usage: python pbc_circuit_reader.py <pbc_file_path>")
        sys.exit(1)

    filepath = sys.argv[1]

    # Validate the file
    is_valid, errors = validate_pbc_file(filepath)

    if is_valid:
        print(f"✓ PBC file is valid: {filepath}")
        print()
        print_pbc_file_summary(filepath)
    else:
        print(f"✗ PBC file has errors: {filepath}")
        print("Errors:")
        for error in errors:
            print(f"  - {error}")